<a href="https://colab.research.google.com/github/MARIEMJ0/BN4601-AI-Cardiology-/blob/app-integration/cardiolens_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!nvidia-smi
!pip install -q gradio pillow pandas pydantic
!pip install -U gradio

#!pip install -r requirements.txt

import torch
import sys
import zipfile
import numpy as np
import gradio as gr
from pathlib import Path
from google.colab import files
!pip install -q "torch>=2.6,<3" "numpy>=1.26"
# import torch.nn as nn
#import torch.optim as optim
#from torchvision import models, transforms
#from collections import Counter
#import matplotlib.pyplot as plt
#from sklearn.metrics import classification_report, confusion_matrix
#from torch.utils.data import Dataset, DataLoader


if torch.cuda.is_available():
  gpu = torch.cuda.get_device_properties(0)
  print("✓ GPU:", gpu.name)
  print(f"✓ VRAM: {gpu.total_memory / 1024**3:.1f} GB")
else:
  print("✗ No GPU detected - enable GPU in Runtime settings")

Tue Nov  3 05:51:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [47]:
# Create a project folder
PROJECT_DIR = Path("/content/cardiolens")
PROJECT_DIR.mkdir(exist_ok=True)

print(PROJECT_DIR)

/content/cardiolens


In [48]:
# upload and extract ZIP
uploaded = files.upload()

zip_path = "/content/ecg_app_handoff.zip"
extract_dir = Path("/content")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

BUNDLE_DIR = Path("/content/ecg_app_handoff")
print(list(BUNDLE_DIR.iterdir()))

Saving ecg_app_handoff.zip to ecg_app_handoff (3).zip
[PosixPath('/content/ecg_app_handoff/source_freeze_manifest.json'), PosixPath('/content/ecg_app_handoff/training_history.json'), PosixPath('/content/ecg_app_handoff/training_selection.json'), PosixPath('/content/ecg_app_handoff/test_fixed05_per_label.csv'), PosixPath('/content/ecg_app_handoff/example_expected_output.json'), PosixPath('/content/ecg_app_handoff/training_environment.txt'), PosixPath('/content/ecg_app_handoff/training_class_weights.csv'), PosixPath('/content/ecg_app_handoff/test_summary.json'), PosixPath('/content/ecg_app_handoff/dataset_README.txt'), PosixPath('/content/ecg_app_handoff/test_prevalence_baseline.json'), PosixPath('/content/ecg_app_handoff/dataset_lead_order.json'), PosixPath('/content/ecg_app_handoff/test_per_label.csv'), PosixPath('/content/ecg_app_handoff/MEMBER_3_HANDOFF.md'), PosixPath('/content/ecg_app_handoff/thresholds.json'), PosixPath('/content/ecg_app_handoff/normalization.npz'), PosixPath('/co

In [49]:
# loading model
sys.path.insert(0, str(BUNDLE_DIR))

from inference import Predictor
print("Loading ECG model...")

predictor = Predictor(
  str(BUNDLE_DIR),
  device="cpu")

print("ECG model loaded.")

# model wrapper
class ECGModelWrapper:
  def __init__(self, predictor):
    self.predictor = predictor

  def analyse(self, ecg_path):
    signal = np.load(
        ecg_path,
        allow_pickle=False)
    result = self.predictor.predict(signal)
    return signal, result

# initialise
ecg_model = ECGModelWrapper(predictor)

Loading ECG model...
ECG model loaded.


In [50]:
FOLLOW_UP_QUESTIONS = [
    "What symptoms are present, and when did they begin?",
    "Are there any previous cardiac diagnoses or relevant medications?",
    "Has the patient experienced similar episodes before?",
    "Is there chest pain, dizziness, fainting, or palpitations?",
    "Are there any relevant risk factors such as hypertension, diabetes, or smoking?",]

In [51]:
import uuid

def create_session():
  return {
      "session_id": str(uuid.uuid4()),
      "ecg_path": None,
      "clinical_history": [],
      "conversation": [],
      "findings": "",
      "evidence": "",
      "explanation": "",
      "confidence": "",
      "follow_up_question": "",
      "pending_follow_up": None,
      "limitations": "",
      "model_result": None,
      "interpretation": "",
      "asked_questions":[],
      "report": ""}

In [52]:
def get_next_question(state):
  for question in FOLLOW_UP_QUESTIONS:
    if question not in state["asked_questions"]:
      state["asked_questions"].append(question)
      return question
  return None

In [53]:
ANNOTATION_EXPLANATIONS = {
    "SR": (
        "Sinus rhythm annotation. This suggests the rhythm was recorded "
        "as originating from the sinus node. It does not prove that the "
        "entire ECG is normal."),
    "AFIB": (
        "Atrial fibrillation annotation. This indicates that the recording "
        "was labelled as having an irregular atrial rhythm."),
    "STACH": (
        "Sinus tachycardia annotation. This indicates a sinus rhythm with "
        "a faster-than-usual rate according to the dataset annotation."),
    "SBRAD": (
        "Sinus bradycardia annotation. This indicates a sinus rhythm with "
        "a slower-than-usual rate according to the dataset annotation."),
    "SARRH": (
        "Sinus arrhythmia annotation. This indicates variation in the "
        "timing of sinus beats."),
    "AFLT": (
        "Atrial flutter annotation. This indicates that the recording was "
        "labelled as showing an atrial flutter rhythm."),
    "SVTAC": (
        "Supraventricular tachycardia annotation. This indicates a fast "
        "rhythm arising above the ventricles."),
    "PSVT": (
        "Paroxysmal supraventricular tachycardia annotation. This indicates "
        "a labelled episode of supraventricular tachycardia."),
    "SVARR": (
        "Supraventricular arrhythmia annotation. This indicates a labelled "
        "rhythm disturbance arising above the ventricles."),
    "PACE": (
        "Pacemaker annotation. This indicates that paced activity was "
        "identified in the recording."),
    "BIGU": (
        "Bigeminy annotation. This indicates a repeating pattern involving "
        "alternating normal and abnormal beats."),
    "TRIGU": (
        "Trigeminy annotation. This indicates a repeating pattern in which "
        "every third beat is abnormal according to the annotation.")}

In [54]:
def explain_model_result(model_result):
  labels = model_result["predicted_annotations"]

  if not labels:
    return ("No supported rhythm annotation exceeded its threshold."
        "This does not prove that the ECG is normal.")

    explanations = []

    for label in labels:
      explanation = ANNOTATION_EXPLANATIONS.get(label,
          "No explanation is available for this annotation.")

      explanations.append(f"{label}: {explanation}")

    return "\n\n".join(explanations)

In [55]:
# initial ecg analysis
def analyse_ecg(ecg_path, initial_history, state):
  print("ECG path:", ecg_path)
  print("Initial history:", initial_history)
  print("State type:", type(state))

  if ecg_path is None:
    return ("Please upload an ECG recording first.","",state)

  # Start a new case
  state = create_session()
  state["ecg_path"] = ecg_path

  # Save the clinician's initial history
  initial_history = initial_history or ""

  if initial_history.strip():
    state["clinical_history"].append({
        "source": "clinician","text": initial_history})

  # Run the waveform model once
  signal, model_result = ecg_model.analyse(ecg_path)

  # Store the raw model result for later questions/reporting
  state["model_result"] = model_result

  predicted_labels = model_result["predicted_annotations"]

  if predicted_labels:
    findings = (
        "Supported rhythm annotations:\n"
        + "\n".join(f"- {label}" for label in predicted_labels))
  else:
    findings = ("No supported rhythm annotation exceeded its threshold.\n\n"
        "This does not prove that the ECG is normal.")

  evidence = ("The findings come from the waveform model's analysis of the "
      "12-lead, 10-second ECG recording.")

  limitations = (model_result["score_note"]
      + "\n\n"
      "This model evaluates supported rhythm annotations only. "
      "It does not provide a definitive diagnosis.")

  explanation = explain_model_result(model_result)

  # For now, this is a placeholder Clinical Detective question.
  follow_up = get_next_question(state)

  state["findings"] = findings
  state["evidence"] = evidence
  state["confidence"] = "Not available"
  state["follow_up_question"] = follow_up
  state["explanation"] = explanation
  state["limitations"] = limitations
  state["pending_follow_up"] = follow_up

  output = (
      f"Findings:\n{findings}\n\n"
      f"Evidence:\n{evidence}\n\n"
      f"Explanation:\n{explanation}\n\n"
      f"Limitations:\n{limitations}")

  state["interpretation"] = output
  state["explanation"] = explanation

  chat_history = [{"role": "assistant","content": output},
    {"role": "assistant","content": follow_up}]

  state["conversation"] = chat_history

  return (chat_history,state)

In [56]:
def start_new_case():
  return create_session(), [], "", ""

In [57]:
def answer_follow_up(answer, state, chat_history):
  if not answer or not answer.strip():
    return (chat_history, state["follow_up_question"], state)

  previous_question = state["follow_up_question"]
  state["clinical_history"].append({
      "source": "clinician",
      "text": answer})

  chat_history = chat_history or []
  # Add only the clinician's answer.
  # The previous AI question is already visible above it in the chat.
  chat_history.append({
    "role": "user",
    "content": answer})

  updated_interpretation = (
        f"Findings:\n"
        f"{state['findings']}\n\n"
        f"Evidence:\n"
        f"{state['evidence']}\n\n"
        f"Clinical context considered:\n"
        f"{answer}\n\n"
        f"Explanation:\n"
        f"{state.get('explanation', '')}\n\n"
        f"Limitation:\n"
        f"{state['limitations']}")

  state["interpretation"] = updated_interpretation

  next_question = get_next_question(state)

  if next_question is None:
    next_question = ("No further follow-up questions are required. "
        "You may generate the report.")

  state["follow_up_question"] = next_question
  state["pending_follow_up"] = next_question
  state["interpretation"] = updated_interpretation

  # Show the next question without unnecessary headings.
  if next_question:
    chat_history.append({
        "role": "assistant",
        "content": next_question})

  return (chat_history,state)

In [58]:
def ask_about_ecg(question, state, chat_history):
  if not question or not question.strip():
    return ("Please enter a question.", state["follow_up_question"], chat_history, state)

  question_lower = question.lower()

  if "what is sr" in question_lower:
    answer = (
        "SR refers to sinus rhythm. It is a rhythm annotation indicating "
        "that the recording was labelled as originating from the sinus "
        "node. It does not mean that the complete ECG is normal.")
  elif "finding" in question_lower:
    answer = state["findings"]
  else:
    answer = (
        "The current waveform model only provides supported rhythm "
        "annotations. It cannot answer questions about ECG features that "
        "it was not trained or evaluated to measure.")

  state["conversation"].append({
        "role": "clinician",
        "content": question})

  state["conversation"].append({
        "role": "assistant",
        "content": answer})

  chat_history = chat_history or []
  chat_history.append({
      "role": "user",
      "content": question})

  chat_history.append({
      "role": "assistant",
      "content": answer})

  return (chat_history, state["follow_up_question"], state)

In [59]:
#Your current teammate handoff does not include a conversation model,
#so this function cannot yet generate intelligent answers.
#Use this temporary version:
def send_message(message_type, message, state, chat_history):
  chat_history = chat_history or []
  if state["ecg_path"] is None:
    chat_history.append({"role": "assistant",
            "content": "Please upload and analyse an ECG first."})
    return ("Please upload and analyse an ECG first.",
        state["follow_up_question"], chat_history, state)

  if not message or not message.strip():
    chat_history.append({"role": "assistant",
            "content": "Please enter an answer or question."})
    return ("Please enter an answer or question.",
        state["follow_up_question"],chat_history, state)

  if message_type == "Answer AI follow-up":
    return answer_follow_up(message, state, chat_history)

  return ask_about_ecg(message, state, chat_history)

In [60]:
# generate a report
def generate_report(state):
  if state["model_result"] is None:
    return "Please analyse an ECG first."

  result = state["model_result"]
  labels = result["predicted_annotations"]

  if labels:
    annotations = ", ".join(labels)
  else:
    annotations = "None exceeded the supplied thresholds."

  history = "\n".join(f"- {item['text']}" for item in state["clinical_history"])

  return f"""
PRELIMINARY AI-ASSISTED ECG REVIEW

Input:
12-lead ECG waveform, 10 seconds, 100 Hz

Model-suggested rhythm annotations:
{annotations}

Clinical context provided:
{history or "- None provided"}

Interpretation scope:
This result covers only the supported rhythm annotations from the waveform model.

Model limitation:
{result["score_note"]}

Remaining question:
{state["follow_up_question"]}

This is an academic AI-assisted prototype and not a definitive diagnosis.
Final interpretation remains the responsibility of the clinician.
"""

In [ ]:
# TO CALL THE VLM
vlm_response = vlm.explain_case(
    ecg_plot_path=ecg_plot_path,
    model_result=model_result,
    clinical_history=state["clinical_history"],
    conversation=state["conversation"])

# VLM SHOULD RECIEVE
{"ecg_model_output": {"predicted_annotations": ["SR"],"scores_are_calibrated": False},
"clinical_history": ["shortness of breath for 2 hours"],
"conversation": [],
"analysis_scope": "Supported rhythm annotations from a 12-lead, 10-second ECG"}


# VLM_ADAPTER.PY SHOULD CONTAIN

class VLMAdapter:

  def __init__(self, model_name_or_path):
    # Load the pretrained VLM once here
    self.model = ...
    self.processor = ...

  def explain_case(self,ecg_plot_path,model_result,clinical_history,conversation):
    prompt = f"""You are assisting a clinician with a preliminary ECG review.

    Waveform-model findings:
    {model_result}

    Clinical history:
    {clinical_history}

    Conversation:
    {conversation}

    Explain the supported findings clearly.
    Do not invent ECG measurements or features that were not provided.
    State uncertainty and limitations.
    Ask one useful follow-up question if information is missing."""

    # Send ECG image and prompt to the VLM
    response = ...

    return {
        "explanation": response,
        "follow_up_question": "...",
        "updated_interpretation": response}

In [61]:
# gradio interface
import inspect

print(gr.__version__)
print(inspect.signature(gr.Chatbot))

with gr.Blocks(title="CardioLens ECG Assistant") as demo:
  state = gr.State(create_session())
  # header
  gr.Markdown(""" # CardioLens ECG Assistant

      AI-assisted preliminary ECG review for clinician use.

      Upload an ECG, review the initial interpretation, answer targeted
      clinical questions, and generate an updated report.""")

  # main workspace
  with gr.Row():
    # left side
    with gr.Column(scale=1, min_width=240):
      new_case_button = gr.Button("+",size="sm",
        variant="secondary",scale=0)

      previous_cases = gr.Dropdown(label="Previous cases",
        choices=[],interactive=True)
      # accepts ECG
      ecg_upload = gr.File(
        label="Upload ECG recording (.npy)",
        file_types=[".npy"],
        type="filepath")

      initial_history = gr.Textbox(
          label="Initial clinical history",
          placeholder="Enter relevant symptoms or history...")

      analyse_button = gr.Button("Analyse ECG", variant="primary")

    # main panel
    with gr.Column(scale=2):

      chatbot = gr.Chatbot(label="Clinical Detective conversation",
      height=300)

      with gr.Row():
        message_type = gr.Radio(
        choices=["Answer AI follow-up","Ask about ECG"],
        value="Answer AI follow-up",
        label="Message type")

        message = gr.Textbox(placeholder="Type your answer or question here...",
              show_label=False, lines=2)

        send_button = gr.Button("Send", variant= "primary")

  # report section
  with gr.Accordion("Report", open=False):
    report_button = gr.Button("Generate report", variant="secondary")

    report = gr.Textbox(label="Generated report",lines=15)

  # event connections
  analyse_button.click(fn=analyse_ecg,
  inputs=[ecg_upload,initial_history,state],
  outputs=[chatbot,state])

  send_button.click(fn=send_message,
  inputs=[message_type, message,state,chatbot],
  outputs=[chatbot,state])

  new_case_button.click(
  fn=start_new_case,
  inputs=None,
  outputs=[state,chatbot,report])

  report_button.click(
  fn=generate_report,
  inputs=[state],
  outputs=[report])

6.31.0
(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = None, watermark: 'str | None' = None,

In [62]:
# launch app in Colab
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a504c6675e43b43aa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
